# 7.26 Pose estimation and keypoints

Pose estimation turns a person from a blob of pixels into a small graph of joints, each found by asking where its heatmap is most confident.

Heatmaps preserve the image grid, Gaussian targets make near misses less wrong, and skeleton geometry turns isolated peaks into a body graph.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
np.set_printoptions(precision=3, suppress=True)

## The concept, built once

For each joint heatmap, pose uses

$$\hat p_k=\operatorname*{arg\,max}_{(y,x)}H_k(y,x).$$

We verify peak $(1,3)$, neighbor value $0.607$, soft coordinate $3.2$, stride mapping, and limb lengths.

In [ ]:
def gaussian_heatmap(size, y0, x0, sigma=1.0):
    yy, xx = np.mgrid[0:size, 0:size]
    return np.exp(-((xx - x0) ** 2 + (yy - y0) ** 2) / (2 * sigma ** 2))


def hard_argmax(heatmap):
    idx = np.argmax(heatmap)
    return np.unravel_index(idx, heatmap.shape)


def soft_argmax_1d(columns, probs):
    return float(np.sum(np.array(columns) * np.array(probs)))


def predict_keypoints(heatmaps):
    return np.array([hard_argmax(hm) for hm in heatmaps])


heatmap = gaussian_heatmap(5, 1, 3)
peak = hard_argmax(heatmap)
neighbor = heatmap[1, 2]
soft_coord = soft_argmax_1d([2, 3, 4], [0.1, 0.6, 0.3])
stride_point = ((1 + 0.5) * 4, (3 + 0.5) * 4)
limbs = [np.linalg.norm(np.array([1, 3]) - np.array([1, 1])), np.linalg.norm(np.array([1, 5]) - np.array([1, 3]))]
assert peak == (1, 3)
assert round(float(neighbor), 3) == 0.607
assert round(soft_coord, 1) == 3.2
assert stride_point == (6.0, 14.0)
assert limbs == [2.0, 2.0]
print("peak", peak)
print("neighbor", round(float(neighbor), 3))
print("soft coordinate", soft_coord)
print("stride center", stride_point)
print("limbs", limbs)

## Dataset ladder

The ladder uses synthetic stick figures with rising jitter, weaker heatmaps, distractor peaks, and occlusion. The metric is mean keypoint error in heatmap pixels.

In [ ]:
def make_pose_ladder():
    specs = [
        ("D1 clean arm", 0.00, 0.0, 1.0),
        ("D2 small jitter", 0.05, 0.0, 0.9),
        ("D3 noisy heatmaps", 0.10, 0.1, 0.8),
        ("D4 distractor", 0.12, 0.25, 0.75),
        ("D5 occluded wrist", 0.18, 0.45, 0.65),
    ]
    base_points = np.array([[2, 3], [4, 5], [6, 7]])
    rungs = []
    for seed, spec in enumerate(specs):
        name, noise, distractor, strength = spec
        local_rng = np.random.default_rng(seed + 40)
        heatmaps = []
        for point in base_points:
            hm = strength * gaussian_heatmap(10, point[0], point[1], sigma=1.0)
            hm = hm + local_rng.normal(0.0, noise, size=hm.shape)
            if distractor > 0:
                hm = hm + distractor * gaussian_heatmap(10, 8 - point[0], 8 - point[1], sigma=0.9)
            heatmaps.append(np.clip(hm, 0.0, None))
        rungs.append({"name": name, "heatmaps": np.array(heatmaps), "true": base_points})
    return rungs


pose_rungs = make_pose_ladder()
for rung in pose_rungs:
    print(rung["name"], "heatmaps", rung["heatmaps"].shape, "keypoints", rung["true"].tolist())

fig, axes = plt.subplots(1, 5, figsize=(12, 2.4))
for ax, rung in zip(axes, pose_rungs):
    ax.imshow(rung["heatmaps"].max(axis=0), cmap="magma")
    ax.set_title(rung["name"])
    ax.axis("off")
plt.tight_layout()

In [ ]:
pose_results = []
for rung in pose_rungs:
    pred = predict_keypoints(rung["heatmaps"])
    errors = np.linalg.norm(pred - rung["true"], axis=1)
    pose_results.append({"name": rung["name"], "pred": pred, "error": float(errors.mean())})

print("rung                 mean keypoint error")
for result in pose_results:
    print(f"{result['name']:<20} {result['error']:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, rung in enumerate(pose_rungs):
    axes[0, i].imshow(rung["heatmaps"].max(axis=0), cmap="magma")
    pred = pose_results[i]["pred"]
    axes[0, i].plot(pred[:, 1], pred[:, 0], "co-")
    axes[0, i].set_title(rung["name"])
    axes[0, i].axis("off")

scores = [item["error"] for item in pose_results]
axes[1, 0].plot(range(1, 6), scores, marker="o")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("mean keypoint error")
axes[1, 0].set_title("localization error")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on D5: row-column swaps and stride mistakes

A heatmap peak is reported as `(y, x)`. Swapping it silently moves every joint.

In [ ]:
hard = pose_rungs[-1]
correct = predict_keypoints(hard["heatmaps"])
swapped = correct[:, ::-1]
correct_error = np.linalg.norm(correct - hard["true"], axis=1).mean()
swapped_error = np.linalg.norm(swapped - hard["true"], axis=1).mean()
image_points = (correct + 0.5) * 4
print("correct heatmap error", round(float(correct_error), 2))
print("swapped heatmap error", round(float(swapped_error), 2))
print("stride-center image points", image_points.tolist())

## Evaluate it + Practice

- Metric: mean keypoint error; no-skill baseline is always predicting the image center.
- Sanity check: D1 peaks should equal the synthetic joints.
- Ablation: remove Gaussian spread and near misses stop being informative.
- Failure signal: high confidence at a geometrically impossible wrist.

Practice:
1. Add an ankle keypoint.
2. Change the heatmap stride from 4 to 2.
3. Make the distractor stronger and watch the error.